# 01 - Data Overview: Exploratory survey of raw data `auto-mpg.csv`

**Contents:**
1. Background & problem objective

2. Data dictionary — meaning, unit, plausible value range for each column

3. Load data & general overview (`head`, `tail`, `sample`, `shape`, `info`, `describe`, memory usage)

4. Data types & various missing-value token formats

5. Unique value count (cardinality) per column

6. Identifying specific "dirty" issues (mixed units, word-written numbers, mixed labels, inconsistent year format,
   European decimal separator, negative/unrealistic values, extra whitespace, duplicates, near-empty rows)

7. Summary table of all issues + suggested preliminary treatment

8. Conclusion & next steps

## 1. Background & problem objective

- **Dataset:** the classic **Auto MPG** dataset (UCI),
  containing technical specifications of automobiles and their fuel consumption (`mpg`).

- **Problem:** regression — predict `mpg` (miles per gallon) from the remaining numeric technical variables.

- **Algorithm used:** Polynomial Regression implemented from scratch (without using
  `sklearn`)

- **Role of the data processing stage:** turn this raw, dirty
  data into clean `X_train/y_train`, `X_val/y_val` (imputed / outliers clipped / standardised),
  and provide a `transform()` function so `X_test/y_test` can be generated whenever needed, ensuring **no
  data leakage** between train/val/test.

## 2. Data Dictionary

**Physical meaning, unit, and plausible value range** for each column — helps identify outliers/unreasonable values in later steps
(based on both pure statistics and domain knowledge).

In [1]:
import pandas as pd

data_dict = pd.DataFrame([
    {'column': 'mpg',          'meaning': 'Fuel consumption target (miles per gallon) - TARGET VARIABLE', 'expected_dtype': 'float', 'unit': 'mpg',    'plausible_range': '~9 - 47'},
    {'column': 'cylinders',    'meaning': 'Number of engine cylinders',                                    'expected_dtype': 'int',   'unit': 'count','plausible_range': '3, 4, 5, 6, 8'},
    {'column': 'displacement', 'meaning': 'Engine displacement',                                           'expected_dtype': 'float', 'unit': 'cubic inch','plausible_range': '~68 - 455'},
    {'column': 'horsepower',   'meaning': 'Engine power',                                                  'expected_dtype': 'float', 'unit': 'hp',      'plausible_range': '~46 - 230'},
    {'column': 'weight',       'meaning': 'Vehicle weight',                                                'expected_dtype': 'float', 'unit': 'pounds (lbs)', 'plausible_range': '~1613 - 5140'},
    {'column': 'acceleration', 'meaning': '0-60 mph acceleration time',                                    'expected_dtype': 'float', 'unit': 'seconds', 'plausible_range': '~8 - 25'},
    {'column': 'model year',   'meaning': 'Model year of the vehicle (2 digits, e.g. 70 = 1970)',           'expected_dtype': 'int',   'unit': 'year',    'plausible_range': '70 - 82 (i.e. 1970-1982)'},
    {'column': 'origin',       'meaning': 'Vehicle origin: 1=USA, 2=Europe, 3=Japan (categorical variable)', 'expected_dtype': 'int',   'unit': 'category code', 'plausible_range': '{1, 2, 3}'},
    {'column': 'car name',     'meaning': 'Car name - identifier, NOT used as a predictive feature',        'expected_dtype': 'string','unit': '-',       'plausible_range': '-'},
])
data_dict

,column,meaning,expected_dtype,unit,plausible_range
0,mpg,Fuel consumption target (miles per gallon) - T...,float,mpg,~9 - 47
1,cylinders,Number of engine cylinders,int,count,"3, 4, 5, 6, 8"
2,displacement,Engine displacement,float,cubic inch,~68 - 455
3,horsepower,Engine power,float,hp,~46 - 230
4,weight,Vehicle weight,float,pounds (lbs),~1613 - 5140
5,acceleration,0-60 mph acceleration time,float,seconds,~8 - 25
6,model year,"Model year of the vehicle (2 digits, e.g. 70 =...",int,year,70 - 82 (i.e. 1970-1982)
7,origin,"Vehicle origin: 1=USA, 2=Europe, 3=Japan (cate...",int,category code,"{1, 2, 3}"
8,car name,"Car name - identifier, NOT used as a predictiv...",string,-,-


## 3. Load data & general overview (`head`, `tail`, `sample`, `shape`, `info`, `describe`)

At this step, data is read the way pandas infers types by default, just to get an initial
general view. Because the data is "dirty" (mixed text, mixed units...), many columns that should
be numeric may end up being read as string type (`object`) — this is itself a sign of dirtiness worth noting.

In [2]:
import pandas as pd
import numpy as np
import re

# Display settings for easier reading when printing DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 60)

RAW_PATH = '../data_raw/auto-mpg.csv'

In [3]:
# Read the raw data with pandas defaults
df_overview = pd.read_csv(RAW_PATH)

print(f'Rows: {df_overview.shape[0]}, Columns: {df_overview.shape[1]}')
df_overview.head(10)

Rows: 407, Columns: 9


,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,33.5,4,85.0,70,-1945,16.8,77.0,3,Datsun F-10 Hatchback
1,35.7,4,98.0,80,1915,14.4,79.0,USA,Dodge Colt Hatchback Custom
2,36.0,4,135.0,84,2370,13.0,82.0,1,dodge charger 2.2
3,32.9,4,119.0,100,2615,14.8,81.0,3,DATSUN 200SX
4,32.0,4,91.0,67,1965,15.7,82.0,3,honda civic (auto)
5,18.5,6,250.0,98,3525,19.0,77.0,1,ford granada
6,20.6,6,225.0,110,3360,16.6,79.0,1,dodge aspen 6
7,33.5,4,151.0,90,2556,13.2,79.0,1,PONTIAC PHOENIX
8,28.0,4,140.0,90,2264,15.5,71.0,1,Chevrolet Vega 2300
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Look at the last few rows and a few random rows (head() isn't enough to assess the whole)
print('Last 5 rows:')
display(df_overview.tail(5))

print('\n5 random rows (random_state=42 for reproducibility):')
display(df_overview.sample(5, random_state=42))

Last 5 rows:


,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
402,16.5,6,168.0,120,3820,16.7,76.0,2,mercedes-benz 280s
403,11.0,8,429.0,208,4633,11.0,72.0,1,mercury marquis
404,10.0,8,360.0,215,4615,14.0,70.0,1,Ford F250
405,24.5,4,98.0,60,2164,22.1,76.0,1,CHEVROLET WOODY
406,29.0,4,90.0,70,1937,14.0,75.0,2,VOLKSWAGEN RABBIT



5 random rows (random_state=42 for reproducibility):


,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
70,26.0,4,97.0,75,2265,18.2,1977.0,3,Toyota Corolla Liftback
218,14.0,8,455.0,225,4425,10.0,70.0,1,pontiac catalina
258,29.0,4,68.0,850,1867,19.5,73.0,2,fiat 128
33,37.0,4,119.0,92,2434,15.0,80.0,3,Datsun 510 Hatchback
42,23.9,8,260.0,90,3420,22.2,79.0,1,Oldsmobile Cutlass Salon Brougham


In [5]:
# General info: column names, non-null counts, dtypes inferred by pandas
df_overview.info()

<class 'pandas.DataFrame'>
RangeIndex: 407 entries, 0 to 406
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           400 non-null    str    
 1   cylinders     404 non-null    str    
 2   displacement  404 non-null    float64
 3   horsepower    395 non-null    str    
 4   weight        404 non-null    str    
 5   acceleration  404 non-null    str    
 6   model year    404 non-null    float64
 7   origin        404 non-null    str    
 8   car name      404 non-null    str    
dtypes: float64(2), str(7)
memory usage: 28.7 KB


In [6]:
# Descriptive statistics for ALL columns (including string columns) to spot initial anomalies
df_overview.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
mpg,400,130,13.0,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cylinders,404,7,4,202,NaN,NaN,NaN,NaN,NaN,NaN,NaN
displacement,404.0,NaN,NaN,NaN,193.87995,104.075326,68.0,105.0,151.0,263.25,455.0
horsepower,395,96,90,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weight,404,352,1985,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
acceleration,404,100,14.5,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model year,404.0,NaN,NaN,NaN,128.314356,312.995299,70.0,73.0,76.0,79.0,2199.0
origin,404,5,1,244,NaN,NaN,NaN,NaN,NaN,NaN,NaN
car name,404,340,ford pinto,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Memory usage - additional reference info, not required but standard
# for the Data Overview step (a dataset this small doesn't affect processing speed)
memory_usage = df_overview.memory_usage(deep=True)
print('Memory usage per column (bytes):')
print(memory_usage)
print(f"\nTotal memory usage: {memory_usage.sum() / 1024:.1f} KB")

Memory usage per column (bytes):
Index             132
mpg             21435
cylinders       20309
displacement     3256
horsepower      20708
weight          21551
acceleration    21502
model year       3256
origin          20329
car name        26616
dtype: int64

Total memory usage: 155.4 KB


## 4. Data types & various missing-value token formats

If the file is read with the default `pd.read_csv()`, pandas will **automatically** convert familiar
tokens (`NA`, `N/A`, `null`, empty cell...) into real `NaN` right at read time, so we **can't see**
what form they were originally recorded in in the raw file.

=> To accurately survey **the different missing-value forms that exist**, we re-read the data with
`dtype=str, keep_default_na=False, na_values=[]` to keep every value as a raw string,
without letting pandas infer anything.

In [8]:
# Number of rows, number of columns
n_rows, n_cols = df_overview.shape
print(f'Rows: {n_rows}')
print(f'Columns: {n_cols}')

# Dtype of each column (as inferred by the default pandas read)
print('\nDtype of each column (pandas-inferred):')
print(df_overview.dtypes)

Rows: 407
Columns: 9

Dtype of each column (pandas-inferred):
mpg                 str
cylinders           str
displacement    float64
horsepower          str
weight              str
acceleration        str
model year      float64
origin              str
car name            str
dtype: object


In [9]:
# Re-read the data KEEPING the raw string form (don't let pandas auto-convert tokens to NaN)
# -> to accurately survey the missing-value forms that exist in the original file
df_str = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False, na_values=[])

# List of tokens commonly used to represent "missing data"
missing_tokens = ['?', 'NA', 'N/A', 'na', 'n/a', 'unknown', 'Unknown', 'UNKNOWN',
                   'nan', 'NaN', 'NAN', 'null', 'NULL', 'None', 'none', '-', '--', '']

def count_missing_forms(df, tokens):
    """Count how many times each missing-value form appears in each column."""
    results = []
    for col in df.columns:
        s = df[col].astype(str).str.strip()
        for tok in tokens:
            count = int((s == tok).sum())
            if count > 0:
                label = 'empty string' if tok == '' else tok
                results.append({'column': col, 'missing_form': label, 'count': count})
    return pd.DataFrame(results)

missing_summary = count_missing_forms(df_str, missing_tokens)
print('Missing-value forms detected, by column:')
missing_summary.sort_values(['column', 'count'], ascending=[True, False]).reset_index(drop=True)

Missing-value forms detected, by column:


,column,missing_form,count
0,acceleration,empty string,3
1,car name,empty string,3
2,cylinders,empty string,3
3,displacement,empty string,3
4,horsepower,?,8
5,horsepower,empty string,6
6,horsepower,NA,3
7,horsepower,N/A,3
8,horsepower,unknown,3
9,model year,empty string,3


In [10]:
# Total missing cells (any form) per column - for easier comparison of impact
missing_total_by_column = (
    missing_summary.groupby('column')['count'].sum()
    .reindex(df_str.columns, fill_value=0)
    .rename('total_missing')
    .to_frame()
)
missing_total_by_column['pct_missing'] = (missing_total_by_column['total_missing'] / n_rows * 100).round(2)
missing_total_by_column

,total_missing,pct_missing
mpg,7,1.72
cylinders,3,0.74
displacement,3,0.74
horsepower,23,5.65
weight,3,0.74
acceleration,3,0.74
model year,3,0.74
origin,3,0.74
car name,3,0.74


## 5. Unique value count (cardinality) per column

Checking **cardinality** (number of unique values)
helps quickly distinguish which columns are continuous variables (many distinct values, e.g. `mpg`, `weight`) and which
columns are actually categorical/discrete variables (few distinct values, e.g. `cylinders`, `origin`,
`model year`) — this is also another way to spot dirty data: if the number of unique
values of a variable that "should only have a few codes" is unexpectedly high, it's likely caused by mixed
representation formats for the same value (exactly as in the `origin`, `cylinders` cases below).

In [11]:
cardinality = pd.DataFrame({
    'n_unique': df_str.nunique(),
    'unique_pct': (df_str.nunique() / n_rows * 100).round(1),
}).sort_values('n_unique')
cardinality

,n_unique,unique_pct
origin,6,1.5
cylinders,8,2.0
model year,24,5.9
displacement,83,20.4
horsepower,99,24.3
acceleration,101,24.8
mpg,131,32.2
car name,341,83.8
weight,353,86.7


In [12]:
# For columns with LOW cardinality (suspected categorical/discrete: cylinders, origin,
# model year) - print ALL unique values to immediately see the mixed representation forms
for col in ['cylinders', 'origin', 'model year']:
    values = sorted(df_str[col].astype(str).str.strip().unique().tolist())
    print(f"{col} ({len(values)} unique values): {values}")
    print()

cylinders (8 unique values): ['', '3', '4', '5', '6', '8', 'four', 'six']

origin (6 unique values): ['', '1', '2', '3', 'Europe', 'USA']

model year (24 unique values): ['', '1970.0', '1971.0', '1972.0', '1974.0', '1976.0', '1977.0', '1978.0', '1979.0', '1982.0', '2199.0', '70.0', '71.0', '72.0', '73.0', '74.0', '75.0', '76.0', '77.0', '78.0', '79.0', '80.0', '81.0', '82.0']



## 6. Identifying specific data quality issues

All operations use `df_str` (data kept as raw strings, not auto-interpreted by
pandas) to ensure issues are detected correctly and completely, then illustrated with a few example rows.


### 6.1. Column `weight`: mixed units (string "lbs" appended to numeric values)

In [13]:
mask_weight_lbs = df_str['weight'].astype(str).str.contains('lbs', case=False, na=False)

print(f'Rows where weight contains the "lbs" unit: {mask_weight_lbs.sum()}')
df_str.loc[mask_weight_lbs, ['weight']].head(3)

Rows where weight contains the "lbs" unit: 10


,weight
53,3433 lbs
172,1845 lbs
183,2200 lbs


### 6.2. Column `weight`: negative values (physically impossible weight)

In [14]:
weight_numeric = pd.to_numeric(
    df_str['weight'].astype(str).str.replace('lbs', '', case=False, regex=False).str.strip(),
    errors='coerce',
)
mask_weight_negative = weight_numeric < 0

print(f'Rows with negative weight: {mask_weight_negative.sum()}')
df_str.loc[mask_weight_negative.fillna(False), ['weight']]

Rows with negative weight: 3


,weight
0,-1945
226,-2506
266,-2605


### 6.3. Column `cylinders`: numbers written as words (four, six, eight...)

In [15]:
number_words = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']
mask_cylinders_word = df_str['cylinders'].astype(str).str.strip().str.lower().isin(number_words)

print(f'Rows where cylinders is written as a word: {mask_cylinders_word.sum()}')
df_str.loc[mask_cylinders_word, ['cylinders']].head(3)

Rows where cylinders is written as a word: 5


,cylinders
169,six
208,six
282,four


### 6.4. Column `origin`: mixed labels — numeric (1/2/3) vs. word (USA/Europe/Japan)

In [16]:
mask_origin_word = df_str['origin'].astype(str).str.strip().str.match(r'^[A-Za-z]+$', na=False)
mask_origin_numeric = df_str['origin'].astype(str).str.strip().str.match(r'^\d+$', na=False)

print(f'Rows where origin is a word (e.g. USA/Europe/Japan): {mask_origin_word.sum()}')
print(f'Rows where origin is numeric (e.g. 1/2/3): {mask_origin_numeric.sum()}')
print('\nUnique origin values appearing in the data:')
print(sorted(df_str['origin'].astype(str).str.strip().unique().tolist()))

df_str.loc[mask_origin_word, ['origin']].head(3)

Rows where origin is a word (e.g. USA/Europe/Japan): 12
Rows where origin is numeric (e.g. 1/2/3): 392

Unique origin values appearing in the data:
['', '1', '2', '3', 'Europe', 'USA']


,origin
1,USA
41,Europe
88,USA


### 6.5. Column `model year`: inconsistent year format (2-digit vs 4-digit)

The actual values in the file have a decimal suffix (e.g. `"70.0"`, `"1970.0"`), not plain
integers — so the checking regex must also account for the `.0` suffix, otherwise the count will be wrong (0 matching rows).

In [17]:
model_year_str = df_str['model year'].astype(str).str.strip()
mask_year_2digit = model_year_str.str.match(r'^\d{2}(\.0)?$', na=False)   # e.g.: '70' or '70.0'
mask_year_4digit = model_year_str.str.match(r'^\d{4}(\.0)?$', na=False)   # e.g.: '1970' or '1970.0'

print(f'Rows where model year is 2 digits (e.g. 70 / 70.0): {mask_year_2digit.sum()}')
print(f'Rows where model year is 4 digits (e.g. 1970 / 1970.0): {mask_year_4digit.sum()}')

pd.concat([
    df_str.loc[mask_year_2digit, ['model year']].head(2),
    df_str.loc[mask_year_4digit, ['model year']].head(2),
])

Rows where model year is 2 digits (e.g. 70 / 70.0): 393
Rows where model year is 4 digits (e.g. 1970 / 1970.0): 11


,model year
0,77.0
1,79.0
43,1976.0
54,1970.0


### 6.6. Column `model year`: unrealistic values (e.g. year `2199`)

The original Auto MPG dataset only contains vehicles produced between 1970-1982 — a value like `2199`
is clearly a data-entry error, not an actual production year.

In [18]:
model_year_num = pd.to_numeric(model_year_str, errors='coerce')
# convert 2-digit years to 4-digit before checking plausibility (e.g. 70 -> 1970); leave 4-digit years unchanged
model_year_normalized = model_year_num.where(model_year_num >= 100, model_year_num + 1900)
mask_year_unrealistic = (model_year_normalized < 1960) | (model_year_normalized > 2030)

print(f'Rows with unrealistic model year: {mask_year_unrealistic.sum()}')
df_str.loc[mask_year_unrealistic.fillna(False), ['model year']]

Rows with unrealistic model year: 1


,model year
177,2199.0


### 6.7. Column `acceleration`: European decimal comma (e.g. "14,5" instead of "14.5")

In [19]:
mask_accel_comma = df_str['acceleration'].astype(str).str.contains(',', na=False)

print(f'Rows where acceleration uses a decimal comma: {mask_accel_comma.sum()}')
df_str.loc[mask_accel_comma, ['acceleration']].head(3)

Rows where acceleration uses a decimal comma: 6


,acceleration
22,"19,5"
169,"17,6"
202,"14,5"


### 6.8. Column `horsepower`: negative values (engine power cannot be negative)

In [20]:
hp_numeric = pd.to_numeric(df_str['horsepower'], errors='coerce')

mask_hp_negative = hp_numeric < 0
mask_hp_too_large = hp_numeric >= 1000   # in case of values like 9999

print(f'Rows with negative horsepower: {mask_hp_negative.sum()}')
print(f'Rows with excessively large horsepower (>= 1000): {mask_hp_too_large.sum()}')
df_str.loc[mask_hp_negative.fillna(False), ['horsepower']]

Rows with negative horsepower: 2
Rows with excessively large horsepower (>= 1000): 0


,horsepower
176,-50
282,-50


### 6.9. Column `car name`: leading/trailing whitespace

`car name` is not used as a predictive feature, but extra whitespace still needs to be recorded because it can
distort duplicate detection (two rows for the same car, where one has extra whitespace, will
NOT be recognised as a duplicate by `duplicated()`).

In [21]:
mask_car_name_ws = df_str['car name'] != df_str['car name'].str.strip()

print(f'Rows where car name has leading/trailing whitespace: {mask_car_name_ws.sum()}')
df_str.loc[mask_car_name_ws, ['car name']].head(3)

Rows where car name has leading/trailing whitespace: 57


,car name
3,DATSUN 200SX
7,PONTIAC PHOENIX
34,TOYOTA CORONA MARK II


### 6.10. Duplicate rows

In [22]:
mask_dup = df_str.duplicated(keep=False)   # keep=False -> flags ALL duplicate records (including the first one)
mask_dup_extra = df_str.duplicated(keep='first')  # only counts records AFTER the first as "extra"

print(f'Total rows belonging to a duplicate group: {mask_dup.sum()}')
print(f'"Extra" duplicate rows (if one original copy is kept): {mask_dup_extra.sum()}')
df_str.loc[mask_dup].sort_values(list(df_str.columns)).head(6)

Total rows belonging to a duplicate group: 5
"Extra" duplicate rows (if one original copy is kept): 3


,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
9,,,,,,,,,
98,,,,,,,,,
279,,,,,,,,,
199,20.2,6,232.0,90,3265,18.2,79.0,1,amc concord dl 6
227,20.2,6,232.0,90,3265,18.2,79.0,1,amc concord dl 6


### 6.11. Near-empty rows (>= 80% of columns missing)

In [23]:
# Since df_str keeps the raw string form, the missing tokens (listed in section 4) need to be treated as NaN when counting
df_check_empty = df_str.replace(
    {tok: np.nan for tok in missing_tokens if tok != ''}
).replace('', np.nan)

missing_count_per_row = df_check_empty.isna().sum(axis=1)
threshold = int(np.ceil(n_cols * 0.8))  # >= 80% of columns missing is considered "near-empty"
mask_near_empty = missing_count_per_row >= threshold

print(f'Threshold for "near-empty": missing >= {threshold}/{n_cols} columns')
print(f'Rows that are near-empty (mostly NaN): {mask_near_empty.sum()}')
df_str.loc[mask_near_empty].head(3)

Threshold for "near-empty": missing >= 8/9 columns
Rows that are near-empty (mostly NaN): 3


,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
9,,,,,,,,,
98,,,,,,,,,
279,,,,,,,,,


## 7. Summary table of all identified issues

Consolidates all the issues surveyed above into a single table, with suggested preliminary treatment
(the actual treatment will be carried out in later notebooks, based on parameters fit from `train_raw`).

In [24]:
summary_table = pd.DataFrame([
    {
        'column': 'multiple columns',
        'issue_type': 'Missing values (multiple forms: ?, NA, unknown, empty string, NaN...)',
        'count': int(missing_summary['count'].sum()),
        'suggested_treatment': 'Standardise all to NaN, then impute (median/mode) - fit on train_raw',
    },
    {
        'column': 'weight',
        'issue_type': 'Mixed units (contains the string "lbs" in numeric values)',
        'count': int(mask_weight_lbs.sum()),
        'suggested_treatment': 'Strip the "lbs" suffix, cast to numeric (unified unit: pounds)',
    },
    {
        'column': 'weight',
        'issue_type': 'Unrealistic negative value',
        'count': int(mask_weight_negative.sum()),
        'suggested_treatment': 'Treat as invalid -> NaN, handle as missing (impute from train)',
    },
    {
        'column': 'cylinders',
        'issue_type': 'Numbers written as words (four, six, eight...)',
        'count': int(mask_cylinders_word.sum()),
        'suggested_treatment': 'Map word -> number (four->4, six->6...), cast to int',
    },
    {
        'column': 'origin',
        'issue_type': 'Mixed labels between numeric (1/2/3) and word (USA/Europe/Japan)',
        'count': int(mask_origin_word.sum()),
        'suggested_treatment': 'Map to a single consistent form (e.g. numeric 1/2/3), fit mapping on train_raw',
    },
    {
        'column': 'model year',
        'issue_type': 'Inconsistent year format (70 vs 1970, both with a .0 suffix)',
        'count': int(mask_year_2digit.sum()),
        'suggested_treatment': 'Normalise to 4-digit year (70 -> 1970)',
    },
    {
        'column': 'model year',
        'issue_type': 'Unrealistic year value (e.g. 2199)',
        'count': int(mask_year_unrealistic.sum()),
        'suggested_treatment': 'Treat as invalid -> NaN, handle as missing (impute from train)',
    },
    {
        'column': 'acceleration',
        'issue_type': 'European-style decimal comma (e.g. 14,5 instead of 14.5)',
        'count': int(mask_accel_comma.sum()),
        'suggested_treatment': 'Replace comma (,) with period (.), cast to float',
    },
    {
        'column': 'horsepower',
        'issue_type': 'Unrealistic negative value',
        'count': int(mask_hp_negative.sum()),
        'suggested_treatment': 'Treat as invalid -> NaN, handle as missing (impute from train)',
    },
    {
        'column': 'car name',
        'issue_type': 'Leading/trailing whitespace',
        'count': int(mask_car_name_ws.sum()),
        'suggested_treatment': 'strip() whitespace; car name is not used as a feature so this only affects duplicate detection',
    },
    {
        'column': 'entire row',
        'issue_type': 'Fully duplicated rows',
        'count': int(mask_dup_extra.sum()),
        'suggested_treatment': 'Remove (drop_duplicates), keep only one record',
    },
    {
        'column': 'entire row',
        'issue_type': 'Near-empty rows (>= 80% of columns missing)',
        'count': int(mask_near_empty.sum()),
        'suggested_treatment': 'Remove before imputing (not enough information to impute meaningfully)',
    },
])
summary_table

,column,issue_type,count,suggested_treatment
0,multiple columns,"Missing values (multiple forms: ?, NA, unknown, empty st...",51,"Standardise all to NaN, then impute (median/mode) - fit ..."
1,weight,"Mixed units (contains the string ""lbs"" in numeric values)",10,"Strip the ""lbs"" suffix, cast to numeric (unified unit: p..."
2,weight,Unrealistic negative value,3,"Treat as invalid -> NaN, handle as missing (impute from ..."
3,cylinders,"Numbers written as words (four, six, eight...)",5,"Map word -> number (four->4, six->6...), cast to int"
4,origin,Mixed labels between numeric (1/2/3) and word (USA/Europ...,12,"Map to a single consistent form (e.g. numeric 1/2/3), fi..."
5,model year,"Inconsistent year format (70 vs 1970, both with a .0 suf...",393,Normalise to 4-digit year (70 -> 1970)
6,model year,Unrealistic year value (e.g. 2199),1,"Treat as invalid -> NaN, handle as missing (impute from ..."
7,acceleration,"European-style decimal comma (e.g. 14,5 instead of 14.5)",6,"Replace comma (,) with period (.), cast to float"
8,horsepower,Unrealistic negative value,2,"Treat as invalid -> NaN, handle as missing (impute from ..."
9,car name,Leading/trailing whitespace,57,strip() whitespace; car name is not used as a feature so...


## 8. Conclusion

**Survey and record** the current state of the dirty data — **nothing has been fixed
in the data yet**. Summary of confirmed findings:

- The dataset contains **407 rows, 9 columns** (8 numeric variables + 1 identifier variable `car name`), the prediction target is `mpg`.

- Missing values appear in **various token formats** across multiple columns (see section 4).

- `cylinders` and `origin` have cardinality **unexpectedly high for a categorical variable** —
  caused by mixed representation formats (word/number) for the same value (see sections 5, 6.3, 6.4).
  
- There are **11 specific data quality issues** identified with affected row counts and examples
  (mixed units, word-written numbers, mixed labels, inconsistent year format, unrealistic years,
  European decimal separator, negative `weight`/`horsepower`, trailing whitespace, duplicate rows,
  near-empty rows) — see the summary table in section 7.